In [3]:
!pip install --upgrade transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 84.0 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [53]:
from transformers import AutoModelForCausalLM

MODEL_ID = "google/gemma-4-E2B"
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype="auto",
    device_map="auto"
)

Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

In [2]:
from transformers import AutoTokenizer

# Load the tokenizer for a specific Gemma model
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

tokenizer_config.json:   0%|          | 0.00/906 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

In [3]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [30]:
import torch.nn.init as init

class GemmaForMCQs(torch.nn.Module):
    ID_TO_LABEL = {
        0: 'A',
        1: 'B',
        2: 'C',
        3: 'D',
        4: 'E'
    }

    def __init__(self, model):
        super().__init__()
        self.model = model
        self.gemma_vocab_size = 262144
        self.labels = 5 # We have 5 options for MCQs
        self.out_layer = torch.nn.Linear(self.gemma_vocab_size, self.labels, device=device, dtype=torch.bfloat16)
        init.xavier_uniform_(self.out_layer.weight)
        init.zeros_(self.out_layer.bias)


    def forward(self, input_ids, targets=None):
        input_ids = input_ids.to(device)
        outputs = self.model(input_ids=input_ids)
        logits = outputs.logits
        last_logits = logits[:,-1,:] # B, 262144
        out = self.out_layer(last_logits) # B, 5

        if targets is not None:
            loss = torch.nn.functional.cross_entropy(out, targets) # B, 1
        else:
            loss = None

        return out, loss

    def generate(self, question, choices):
        prompt = f'<question> {question} \n' + f'<options>\n'
        for id, choice in enumerate(choices):
            prompt = prompt + f'{self.ID_TO_LABEL[id]}. {choice}\n'

        inputs = tokenizer(prompt, return_tensors="pt") # No .to(device) here, as forward handles it.
        mcq_choice_scores, _ = self.forward(input_ids=inputs['input_ids']) # Call the forward method of *this* GemmaForMCQs instance
        predicted_label = torch.argmax(mcq_choice_scores, dim=-1)

        return self.ID_TO_LABEL[predicted_label.item()]

In [54]:
gemma_for_mcq_model = GemmaForMCQs(model)

In [55]:
gemma_for_mcq_model

GemmaForMCQs(
  (model): Gemma4ForConditionalGeneration(
    (model): Gemma4Model(
      (language_model): Gemma4TextModel(
        (embed_tokens): Gemma4TextScaledWordEmbedding(262144, 1536, padding_idx=0)
        (layers): ModuleList(
          (0-3): 4 x Gemma4TextDecoderLayer(
            (self_attn): Gemma4TextAttention(
              (q_proj): Linear(in_features=1536, out_features=2048, bias=False)
              (q_norm): Gemma4RMSNorm()
              (k_norm): Gemma4RMSNorm()
              (v_norm): Gemma4RMSNorm()
              (k_proj): Linear(in_features=1536, out_features=256, bias=False)
              (v_proj): Linear(in_features=1536, out_features=256, bias=False)
              (o_proj): Linear(in_features=2048, out_features=1536, bias=False)
            )
            (mlp): Gemma4TextMLP(
              (gate_proj): Linear(in_features=1536, out_features=6144, bias=False)
              (up_proj): Linear(in_features=1536, out_features=6144, bias=False)
              (down_pr

In [7]:
import math

class LoRALayer(torch.nn.Module):
    def __init__(self, in_dim, out_dim, rank, alpha):
        super().__init__()
        self.A = torch.nn.Parameter(torch.empty(in_dim, rank, dtype=torch.bfloat16, device=device))
        torch.nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = torch.nn.Parameter(torch.zeros(rank, out_dim, dtype=torch.bfloat16, device=device))
        self.alpha = alpha
        self.rank = rank

    def forward(self, x):
        x = (self.alpha / self.rank) * (x @ self.A @ self.B)
        return x

In [8]:
class LinearWithLoRA(torch.nn.Module):
    def __init__(self, linear, rank, alpha):
        super().__init__()
        self.linear = linear
        self.lora = LoRALayer(
            linear.in_features, linear.out_features, rank, alpha
        )

    def forward(self, x):
        return self.linear(x) + self.lora(x)

In [9]:
import torch.nn as nn

class LoRALayerCollection(nn.Module):
    def __init__(self, lora_layers, out_layer):
        super().__init__()
        # Proper registration for serialization
        self.lora_layers = nn.ModuleList(lora_layers)
        self.out_layer = out_layer


In [10]:
def get_default_lora_layers(model, rank, alpha):
    initial_lora_layers = []
    for name, module in model.named_children():
        if isinstance(module, torch.nn.Linear):
            initial_lora_layers.append(
              LoRALayer(
                module.in_features, module.out_features, rank, alpha
              )
            )
        else:
            initial_lora_layers.extend(get_default_lora_layers(module, rank, alpha))

    return initial_lora_layers

default_lora_layers = get_default_lora_layers(model, rank=16, alpha=16)


In [11]:
len(default_lora_layers)

526

In [12]:
# Add one more LoRA layer for the output weights
default_lora_layers.append(
    LoRALayer(
        262144, 5, rank=16, alpha=16
    )
)

In [13]:
gemma_vocab_size = 262144
labels = 5
default_output_layer = torch.nn.Linear(
    gemma_vocab_size,
    labels,
    device=device,
    dtype=torch.bfloat16
)
init.xavier_uniform_(default_output_layer.weight)
init.zeros_(default_output_layer.bias)

default_output_layer_with_lora = LinearWithLoRA(default_output_layer, rank=16, alpha=16)
lora_layer_with_output_layer = LoRALayerCollection(default_lora_layers, default_output_layer_with_lora)

In [14]:
lora_layer_with_output_layer

LoRALayerCollection(
  (lora_layers): ModuleList(
    (0-526): 527 x LoRALayer()
  )
  (out_layer): LinearWithLoRA(
    (linear): Linear(in_features=262144, out_features=5, bias=True)
    (lora): LoRALayer()
  )
)

In [16]:
pretrained_weights = torch.load('/content/lora_layers_collection.pth')

In [17]:
lora_layer_with_output_layer.load_state_dict(pretrained_weights)

<All keys matched successfully>

In [18]:
lora_layer_with_output_layer

LoRALayerCollection(
  (lora_layers): ModuleList(
    (0-526): 527 x LoRALayer()
  )
  (out_layer): LinearWithLoRA(
    (linear): Linear(in_features=262144, out_features=5, bias=True)
    (lora): LoRALayer()
  )
)

## Let's start modifying the Gemma base model architecture with the pre-loaded fine-tuned LoRA weights

In [56]:
gemma_for_mcq_model

GemmaForMCQs(
  (model): Gemma4ForConditionalGeneration(
    (model): Gemma4Model(
      (language_model): Gemma4TextModel(
        (embed_tokens): Gemma4TextScaledWordEmbedding(262144, 1536, padding_idx=0)
        (layers): ModuleList(
          (0-3): 4 x Gemma4TextDecoderLayer(
            (self_attn): Gemma4TextAttention(
              (q_proj): Linear(in_features=1536, out_features=2048, bias=False)
              (q_norm): Gemma4RMSNorm()
              (k_norm): Gemma4RMSNorm()
              (v_norm): Gemma4RMSNorm()
              (k_proj): Linear(in_features=1536, out_features=256, bias=False)
              (v_proj): Linear(in_features=1536, out_features=256, bias=False)
              (o_proj): Linear(in_features=2048, out_features=1536, bias=False)
            )
            (mlp): Gemma4TextMLP(
              (gate_proj): Linear(in_features=1536, out_features=6144, bias=False)
              (up_proj): Linear(in_features=1536, out_features=6144, bias=False)
              (down_pr

In [57]:
lora_layer_with_output_layer.out_layer

LinearWithLoRA(
  (linear): Linear(in_features=262144, out_features=5, bias=True)
  (lora): LoRALayer()
)

In [58]:
# Load the pre-trained weights only for Output layer, leaving LoRA layer behind.
gemma_for_mcq_model.out_layer = lora_layer_with_output_layer.out_layer.linear

In [59]:
gemma_for_mcq_model

GemmaForMCQs(
  (model): Gemma4ForConditionalGeneration(
    (model): Gemma4Model(
      (language_model): Gemma4TextModel(
        (embed_tokens): Gemma4TextScaledWordEmbedding(262144, 1536, padding_idx=0)
        (layers): ModuleList(
          (0-3): 4 x Gemma4TextDecoderLayer(
            (self_attn): Gemma4TextAttention(
              (q_proj): Linear(in_features=1536, out_features=2048, bias=False)
              (q_norm): Gemma4RMSNorm()
              (k_norm): Gemma4RMSNorm()
              (v_norm): Gemma4RMSNorm()
              (k_proj): Linear(in_features=1536, out_features=256, bias=False)
              (v_proj): Linear(in_features=1536, out_features=256, bias=False)
              (o_proj): Linear(in_features=2048, out_features=1536, bias=False)
            )
            (mlp): Gemma4TextMLP(
              (gate_proj): Linear(in_features=1536, out_features=6144, bias=False)
              (up_proj): Linear(in_features=1536, out_features=6144, bias=False)
              (down_pr

In [23]:
class LinearWithPreloadedLoRA(torch.nn.Module):
    def __init__(self, linear, lora_layer):
        super().__init__()
        self.linear = linear
        self.lora = lora_layer

    def forward(self, x):
        return self.linear(x) + self.lora(x)

In [24]:
def replace_linear_with_preloaded_lora(model, lora_layers, cur_index):
    for name, module in model.named_children():
        if isinstance(module, torch.nn.Linear):
            print(f'Initialized with Pre-loaded LoRA Layer: {cur_index}')
            setattr(model, name, LinearWithPreloadedLoRA(module, lora_layers[cur_index]))
            cur_index = cur_index + 1
        else:
            cur_index = replace_linear_with_preloaded_lora(module, lora_layers, cur_index)

    return cur_index

In [60]:
replace_linear_with_preloaded_lora(gemma_for_mcq_model, lora_layer_with_output_layer.lora_layers, 0)

Initialized with Pre-loaded LoRA Layer: 0
Initialized with Pre-loaded LoRA Layer: 1
Initialized with Pre-loaded LoRA Layer: 2
Initialized with Pre-loaded LoRA Layer: 3
Initialized with Pre-loaded LoRA Layer: 4
Initialized with Pre-loaded LoRA Layer: 5
Initialized with Pre-loaded LoRA Layer: 6
Initialized with Pre-loaded LoRA Layer: 7
Initialized with Pre-loaded LoRA Layer: 8
Initialized with Pre-loaded LoRA Layer: 9
Initialized with Pre-loaded LoRA Layer: 10
Initialized with Pre-loaded LoRA Layer: 11
Initialized with Pre-loaded LoRA Layer: 12
Initialized with Pre-loaded LoRA Layer: 13
Initialized with Pre-loaded LoRA Layer: 14
Initialized with Pre-loaded LoRA Layer: 15
Initialized with Pre-loaded LoRA Layer: 16
Initialized with Pre-loaded LoRA Layer: 17
Initialized with Pre-loaded LoRA Layer: 18
Initialized with Pre-loaded LoRA Layer: 19
Initialized with Pre-loaded LoRA Layer: 20
Initialized with Pre-loaded LoRA Layer: 21
Initialized with Pre-loaded LoRA Layer: 22
Initialized with Pre-

527

In [61]:
gemma_for_mcq_model

GemmaForMCQs(
  (model): Gemma4ForConditionalGeneration(
    (model): Gemma4Model(
      (language_model): Gemma4TextModel(
        (embed_tokens): Gemma4TextScaledWordEmbedding(262144, 1536, padding_idx=0)
        (layers): ModuleList(
          (0-3): 4 x Gemma4TextDecoderLayer(
            (self_attn): Gemma4TextAttention(
              (q_proj): LinearWithPreloadedLoRA(
                (linear): Linear(in_features=1536, out_features=2048, bias=False)
                (lora): LoRALayer()
              )
              (q_norm): Gemma4RMSNorm()
              (k_norm): Gemma4RMSNorm()
              (v_norm): Gemma4RMSNorm()
              (k_proj): LinearWithPreloadedLoRA(
                (linear): Linear(in_features=1536, out_features=256, bias=False)
                (lora): LoRALayer()
              )
              (v_proj): LinearWithPreloadedLoRA(
                (linear): Linear(in_features=1536, out_features=256, bias=False)
                (lora): LoRALayer()
              )
     

In [47]:
from datasets import load_from_disk

common_sense_qa_dataset = load_from_disk('/content/common_sense_qa_tokenized_dataset')
common_sense_qa_dataset

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'labels'],
        num_rows: 9741
    })
    validation: Dataset({
        features: ['input_ids', 'labels'],
        num_rows: 1221
    })
    test: Dataset({
        features: ['input_ids', 'labels'],
        num_rows: 1140
    })
})

In [49]:
import torch
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

class MCQDataset(Dataset):
    LABEL_TO_ID = {
        'A': 0,
        'B': 1,
        'C': 2,
        'D': 3,
        'E': 4
    }

    def __init__(self, tokenized_list):
        self.data = tokenized_list['input_ids']
        self.labels = tokenized_list['labels']

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return (
            torch.tensor(self.data[idx], dtype=torch.long, device=device),
            torch.tensor(self.LABEL_TO_ID[self.labels[idx]], dtype=torch.long, device=device)
        )

In [50]:
mcq_validation_dataset = MCQDataset(common_sense_qa_dataset['validation'])

In [51]:
def get_accuracy(dataset):
  correct = 0
  total = 0
  log_interval = 100

  for row in dataset:
    #print(row)
    inputs = row[0].view(1, -1)
    target = row[1]
    gemma_for_mcq_model.eval()

    with torch.no_grad():
      logits, _ = gemma_for_mcq_model.forward(inputs)
      predicted_label = torch.argmax(logits, dim=-1)
      #print(logits)
      #print(predicted_label.item(), target.item())

    if predicted_label.item() == target:
      correct += 1

    if total % log_interval == 0 and total > 0:
      print(f'Steps: {total} Accuracy: {(correct / total) * 100.0}')

    total += 1

  print(f'Accuracy: {(correct / total) * 100.0}')

In [62]:
get_accuracy(mcq_validation_dataset)

Steps: 100 Accuracy: 69.0
Steps: 200 Accuracy: 69.0
Steps: 300 Accuracy: 68.66666666666667
Steps: 400 Accuracy: 67.5
Steps: 500 Accuracy: 67.4
Steps: 600 Accuracy: 68.66666666666667
Steps: 700 Accuracy: 68.42857142857143
Steps: 800 Accuracy: 68.75
Steps: 900 Accuracy: 69.44444444444444
Steps: 1000 Accuracy: 69.6
Steps: 1100 Accuracy: 70.0909090909091
Steps: 1200 Accuracy: 70.08333333333333
Accuracy: 70.10647010647011


In [63]:
gemma_for_mcq_model.generate(
    'The townhouse was a hard sell for the realtor, it was right next to a high rise what?',
    ["suburban development", "apartment building", "bus stop", "michigan", "suburbs"])

'B'

In [64]:
gemma_for_mcq_model.generate(
    'What were the kids doing as they looked up at the sky and clouds?',
    ["ponder", "become adults", "wonder about", "open door", "distracting"]
)

'C'